# Coverage Days Null Analysis

This notebook analyzes the null values in `coverage_days` inside `../data/interim/inventory_enriched.csv` and applies a safe handling strategy.

## Key idea
- Keep the original `coverage_days` column unchanged.
- Add a reason column for missing values.
- Create a resolved numeric version for downstream use.

In this dataset, the nulls are not random. They happen when a product is inactive and its average daily demand is zero, so `coverage_days = stock / avg_daily_demand` is undefined.

In [ ]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f"{x:,.2f}")

path = "../data/interim/inventory_enriched.csv"
df = pd.read_csv(path)

print("Shape:", df.shape)
df.head()

In [ ]:
null_count = df['coverage_days'].isna().sum()
null_pct = df['coverage_days'].isna().mean() * 100

summary = pd.DataFrame({
    'rows': [len(df)],
    'coverage_days_nulls': [null_count],
    'coverage_days_null_pct': [round(null_pct, 2)]
})

summary

In [ ]:
null_rows = df[df['coverage_days'].isna()].copy()

analysis = {
    'null_rows': len(null_rows),
    'null_and_zero_demand': int((null_rows['avg_daily_demand'] == 0).sum()),
    'null_and_inactive': int((null_rows['is_active'] == False).sum()),
    'null_and_stock_status_healthy': int((null_rows['stock_status'] == 'Healthy').sum()),
    'distinct_products_with_nulls': int(null_rows['product_id'].nunique())
}

pd.Series(analysis)

In [ ]:
null_rows[['product_id', 'product_name', 'category', 'type']].drop_duplicates().reset_index(drop=True)

In [ ]:
root_cause_check = pd.crosstab(
    df['is_active'],
    [df['coverage_days'].isna(), df['avg_daily_demand'].eq(0)],
    rownames=['is_active'],
    colnames=['coverage_is_null', 'avg_daily_demand_is_zero'],
    dropna=False
)

root_cause_check

## Interpretation

These nulls are **business-rule nulls**, not data-entry errors:
- `coverage_days` depends on `stock / avg_daily_demand`.
- For these rows, `avg_daily_demand = 0`.
- The same rows are also `is_active = False`.

So using mean or median imputation would be misleading. The safest approach is to preserve the original nulls and create a resolved version for analysis.

In [ ]:
df_handled = df.copy()

inactive_zero_demand_mask = (
    df_handled['coverage_days'].isna()
    & df_handled['avg_daily_demand'].eq(0)
    & df_handled['is_active'].eq(False)
)

df_handled['coverage_days_missing_reason'] = np.where(
    inactive_zero_demand_mask,
    'inactive_product_zero_demand',
    np.where(df_handled['coverage_days'].isna(), 'other_missing_case', 'observed')
)

# Analytically, zero demand means coverage is unbounded.
# We store that as infinity for numeric workflows.
df_handled['coverage_days_resolved'] = df_handled['coverage_days']
df_handled.loc[inactive_zero_demand_mask, 'coverage_days_resolved'] = np.inf

# Optional capped version for ML models or tools that do not accept infinity.
finite_max = df_handled['coverage_days'].max()
df_handled['coverage_days_resolved_capped'] = df_handled['coverage_days_resolved'].replace(np.inf, finite_max + 1)

df_handled.loc[inactive_zero_demand_mask, [
    'product_id', 'avg_daily_demand', 'coverage_days', 'coverage_days_missing_reason',
    'coverage_days_resolved', 'coverage_days_resolved_capped'
]].head()

In [ ]:
df_handled[['coverage_days', 'coverage_days_resolved', 'coverage_days_resolved_capped']].describe(include='all')

In [ ]:
output_path = '../data/interim/inventory_enriched_handled.csv'
df_handled.to_csv(output_path, index=False)
print(f'Saved handled dataset to: {output_path}')

## Recommendation

Use these columns depending on the task:
- `coverage_days`: original business value, keeps nulls.
- `coverage_days_missing_reason`: explains why the value is missing.
- `coverage_days_resolved`: best for analytics where infinite coverage is meaningful.
- `coverage_days_resolved_capped`: best for tools or models that require finite numeric values.